# Weeks 1–4 Assignment: Public Data Workflow and Structure Decision

**DSSA 5102 Data Gathering and Warehousing**  
**Due:** See the LMS  
**Points:** 100  
**Estimated effort:** 3–4 hours  
**Platform:** Google Colab free tier

In this assignment, you will turn a bounded extract from an official public-data portal into a documented, validated, and reusable data product. Your submission must show not only that the code runs, but also why the source, transformations, representation, and conclusions are appropriate.


## Learning objectives

You will demonstrate that you can:

1. evaluate a public source and define an appropriate use;
2. translate a workplace question into a processing specification;
3. retrieve, inspect, clean, and validate public data;
4. document provenance and unresolved quality issues;
5. implement and justify an appropriate data representation;
6. query data with pandas and introductory SQL;
7. save, reload, and verify reproducible artifacts;
8. communicate findings without overstating what the sample supports.


## Scenario

You are supporting a municipal operations team that wants a compact view of service-request activity. The team needs a reproducible analysis of a small NYC 311 sample and a recommendation about how the cleaned records should be represented for reuse.

Your work must distinguish:

- what the public source actually contains;
- what you changed;
- what your validation establishes;
- what remains unknown;
- which conclusions are and are not supported.


## What to submit

Submit these files through the LMS:

1. your completed and executed notebook, renamed `lastname_firstname_weeks1_4.ipynb`;
2. `nyc_311_assignment_clean.csv`;
3. `nyc_311_assignment_metadata.json`;
4. either `nyc_311_assignment.sqlite` **or** `nyc_311_example_document.json`, depending on your representation choice.

Before submission, use **Runtime → Restart session and run all** in Colab. Every required cell must execute in order without an error. Outputs should remain visible in the submitted notebook.


## Academic integrity and AI policy

Follow the course syllabus. AI may be used only for instructor-authorized brainstorming, explanation, or learning support. It may not complete the submitted code, analysis, interpretation, or reflection unless the instructor explicitly authorizes that use.

If you use permitted AI assistance, document:

- the tool and purpose;
- the question or prompt you supplied;
- what you independently verified;
- what you changed or rejected.

Do not enter private, confidential, employer-owned, or personally identifiable data into an AI system.


# Part 0 — Setup and provided retrieval code


In [ ]:
from pathlib import Path
from urllib.parse import urlencode
import json
import sqlite3

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
OUTPUT_DIR = Path("weeks1_4_assignment_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("pandas version:", pd.__version__)
print("output directory:", OUTPUT_DIR.resolve())


## Public dataset

This assignment uses **NYC Open Data: 311 Service Requests from 2020 to Present**.

- [Dataset page](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9)
- [Official API documentation](https://dev.socrata.com/foundry/data.cityofnewyork.us/erm2-nwe9)
- Dataset identifier: `erm2-nwe9`

The provided code requests selected fields, orders the records, limits the response to 1,500 rows, and uses a fixed historical period. Do not increase the limit or request exact addresses, coordinates, or free-text fields.


In [ ]:
API_BASE = "https://data.cityofnewyork.us/resource/erm2-nwe9.csv"
QUERY = {
    "$limit": 1500,
    "$select": (
        "unique_key,created_date,closed_date,agency,complaint_type,"
        "descriptor,incident_zip,borough,status"
    ),
    "$where": (
        "created_date between '2025-02-01T00:00:00.000' "
        "and '2025-02-07T23:59:59.999'"
    ),
    "$order": "created_date, unique_key",
}
DATA_URL = API_BASE + "?" + urlencode(QUERY)

try:
    raw_311 = pd.read_csv(
        DATA_URL,
        dtype={"unique_key": "string", "incident_zip": "string"},
    )
except Exception as error:
    raise RuntimeError(
        "The public-data request failed. Confirm that the Colab runtime has "
        "internet access and retry this cell."
    ) from error

print("retrieved rows:", len(raw_311))
display(raw_311.head())


The retrieval code above is provided and is not graded as student-written code. You remain responsible for explaining its scope and limitations.

# Part 1 — Source evaluation and purpose — 10 points

Complete the source record below. Use the dataset page and API documentation. Write concise, specific answers.

- Publisher and organizational owner:
- Unit of observation:
- Requested time period:
- Selected fields:
- Fields deliberately excluded and why:
- Appropriate workplace use:
- One inappropriate or unsupported use:
- Refresh or update consideration:
- One known limitation:


**Student response:**

<!-- Replace this comment with your source evaluation. -->


# Part 2 — Processing specification — 10 points

Before writing transformation code, provide a 100–175 word specification covering:

- the workplace question;
- the input and unit of observation;
- cleaning and type-conversion rules;
- how missing or invalid values will be handled;
- required output columns;
- at least six validation checks;
- the artifacts you will save.


**Student specification:**

<!-- Replace this comment with your specification. -->


# Part 3 — Inspect before cleaning — 10 points


Write code that displays or reports all of the following:

1. row and column counts;
2. column names and data types;
3. missing-value counts;
4. duplicate `unique_key` count;
5. distinct counts for agency, complaint type, borough, and status;
6. at least five sample rows.

Add two or three sentences below the output identifying the most important quality or interpretation issues you observed.


In [ ]:
# TODO: Inspect raw_311 without modifying it.


**Inspection observations:**

<!-- Replace this comment with two or three sentences. -->


# Part 4 — Clean and document quality issues — 15 points


Create a new DataFrame named `clean_311`. Do not overwrite `raw_311`.

Required transformations:

- rename `unique_key` to `request_id`;
- trim whitespace from text columns;
- standardize agency, borough, and status consistently;
- parse `created_date` and `closed_date` as datetimes with invalid values converted to missing;
- calculate `resolution_hours` when both timestamps exist;
- create `timing_issue` for negative resolution durations;
- create `needs_review` when a required identifier or created timestamp is missing, or a negative duration is present;
- preserve missing values rather than inventing replacements.

Display at least five cleaned rows and the number of records requiring review.


In [ ]:
# TODO: Create clean_311 from a copy of raw_311.


# Part 5 — Validation evidence — 15 points


Create a dictionary named `validation_results` containing Boolean results for at least six checks. It must include:

- the API row limit was respected;
- the raw row count was preserved;
- required output columns exist;
- request identifiers are complete;
- request identifiers are unique;
- created timestamps parsed successfully or failures were flagged;
- every negative duration is flagged;
- status values are displayed and reviewed rather than assumed.

Convert the dictionary to a small evidence table. Use assertions for checks that must pass, but do not assert that real public data has no quality problems.


In [ ]:
# TODO: Create validation_results and display the evidence table.


**Validation interpretation:** Explain what the checks establish and identify at least one thing they do not establish.

<!-- Replace this comment with your interpretation. -->


# Part 6 — Descriptive analysis — 10 points


Produce two summaries:

1. request counts by borough and status;
2. the ten most frequent agency and complaint-type combinations.

Add one additional summary connected to your workplace question. Appropriate choices include missing-value rates, records with closure timestamps, median resolution time, or another defensible aggregate.

Do not rank agencies or neighborhoods as “best” or “worst” using this limited extract.


In [ ]:
# TODO: Create and display three descriptive summaries.


**Findings:** Write two accurate observations and one explicit limitation. Each observation must refer to a displayed result.

<!-- Replace this comment with your findings and limitation. -->


# Part 7 — Representation and query decision — 10 points


Complete **one** of the following tracks. Then justify why it fits the intended use better than the alternative.

## Track A — Relational representation and SQL

- Create at least two related tables, such as `requests` and `agencies`.
- Give each table an appropriate identifier.
- Verify that agency references in the request table resolve.
- Save the tables to `nyc_311_assignment.sqlite`.
- Run one SQL query that joins or groups the tables and supports your workplace question.

## Track B — Nested JSON document

- Select one request and represent it as a nested dictionary.
- Include a request section, agency section, location section, and timing/quality section.
- Convert the document to JSON and save it as `nyc_311_example_document.json`.
- Flatten it with `pd.json_normalize()` to demonstrate how it can return to tabular form.

Both tracks must preserve the request identifier and avoid converting missing values into misleading text.


In [ ]:
# TODO: Implement Track A or Track B.


**Representation justification:** In 100–150 words, explain the intended user, the principal advantage, the principal tradeoff, and why the selected structure fits better than the alternative.

<!-- Replace this comment with your justification. -->


# Part 8 — Provenance, save, reload, and verification — 10 points


Create a provenance table named `provenance_log` with at least four stages:

1. source and retrieval;
2. inspection;
3. cleaning;
4. validation and output.

For each stage, record the input, action, row count, and evidence or known issue.

Then save:

- `clean_311` as `nyc_311_assignment_clean.csv`;
- source information, query parameters, validation results, and limitations as `nyc_311_assignment_metadata.json`.

Reload both files and verify row count, required columns, request identifiers, and metadata fields.


In [ ]:
# TODO: Create provenance_log, save the required files, reload them, and verify them.


# Part 9 — Responsible-use reflection — 10 points


Write 250–400 words addressing all of the following:

1. What question can your extract reasonably answer?
2. How do the row limit, fixed period, missing values, and source updates affect interpretation?
3. Why should request counts or resolution times not automatically be treated as measures of neighborhood need or agency performance?
4. Which fields did the assignment exclude for privacy or minimization reasons?
5. What additional evidence would be required before making an operational decision?
6. What permitted AI assistance did you use and independently verify, or did you use none?


**Student reflection:**

<!-- Replace this comment with your 250–400 word reflection. -->


# Final self-check


In [ ]:
required_names = ["raw_311", "clean_311", "validation_results", "provenance_log"]
missing_names = [name for name in required_names if name not in globals()]

if missing_names:
    print("Incomplete: define these required objects:", missing_names)
else:
    final_checks = {
        "clean data has rows": len(clean_311) > 0,
        "raw rows preserved": len(clean_311) == len(raw_311),
        "request_id exists": "request_id" in clean_311.columns,
        "request IDs are unique": clean_311["request_id"].is_unique,
        "validation has at least six checks": len(validation_results) >= 6,
        "provenance has at least four stages": len(provenance_log) >= 4,
        "clean CSV exists": (OUTPUT_DIR / "nyc_311_assignment_clean.csv").exists(),
        "metadata JSON exists": (OUTPUT_DIR / "nyc_311_assignment_metadata.json").exists(),
    }
    display(pd.Series(final_checks, name="passed").to_frame())
    print("Required checks passed:", sum(final_checks.values()), "/", len(final_checks))


## Submission checklist

Confirm each item before uploading:

- [ ] The notebook was renamed with my name.
- [ ] I restarted the Colab session and ran all cells in order.
- [ ] All required outputs are visible and no required cell has an error.
- [ ] `raw_311` was preserved.
- [ ] Cleaning rules and quality flags are documented.
- [ ] At least six validation checks are displayed.
- [ ] Findings refer to displayed evidence and include limitations.
- [ ] One representation track is complete and justified.
- [ ] The CSV, metadata JSON, and selected representation artifact exist.
- [ ] My AI-use statement follows the syllabus.
- [ ] I did not include confidential, private, or employer-owned information.


# Grading rubric — 100 points

| Criterion | Points | Full-credit evidence |
|---|---:|---|
| Source evaluation and purpose | 10 | Accurate ownership, unit, scope, appropriate use, exclusions, and limitation |
| Processing specification | 10 | Clear input, rules, outputs, checks, and artifacts before implementation |
| Raw-data inspection | 10 | Complete structural and quality inspection with accurate observations |
| Cleaning and quality flags | 15 | Raw data preserved; required transformations and review flags implemented correctly |
| Validation evidence | 15 | Six or more meaningful checks; appropriate assertions; limits explained |
| Descriptive analysis | 10 | Three correct summaries, two supported observations, and one limitation |
| Representation and query decision | 10 | Selected track works, preserves identifiers, and is persuasively justified |
| Provenance and artifact verification | 10 | Four-stage provenance log; required files save, reload, and verify successfully |
| Responsible-use reflection | 10 | Complete, specific, appropriately cautious, and consistent with the AI policy |
| **Total** | **100** | |

Work that runs but lacks interpretation, provenance, or limitations cannot receive full credit. Assertions without displayed evidence are insufficient.
